# Recurrent PPO on Inverted Pendulum

Train a recurrent PPO agent (GRU/LSTM/vanilla RNN) on the inverted pendulum task using Brax.


In [ ]:
#@markdown ## ⚠️ PLEASE NOTE:
#@markdown This colab runs best using a GPU runtime.  From the Colab menu, choose Runtime > Change Runtime Type, then select **'GPU'** in the dropdown.

import functools
import jax
import os

from datetime import datetime
from jax import numpy as jp
import matplotlib.pyplot as plt

from IPython.display import HTML, clear_output

try:
  import brax
except (ImportError, ModuleNotFoundError):
  !pip install git+https://github.com/google/brax.git@main
  clear_output()

from brax import envs
from brax.io import html
from brax.training.agents import recurrent_ppo
from brax.training.agents.recurrent_ppo import networks as rec_nets
from brax.training import types



Select the environment and backend. This demo focuses on the inverted pendulum.


In [ ]:
#@title Load Env { run: "auto" }

env_name = 'inverted_pendulum'  # @param ['inverted_pendulum']
backend = 'positional'  # @param ['positional']

env = envs.get_environment(env_name=env_name, backend=backend)
state = jax.jit(env.reset)(rng=jax.random.PRNGKey(seed=0))

HTML(html.render(env.sys, [state.pipeline_state]))



We configure a recurrent PPO trainer with a small GRU core and short BPTT windows suited for the pendulum task.


The hyperparameters below were chosen to keep this demo fast while showing the recurrent pipeline.


In [ ]:
#@title Training

# Define a network factory so we can reuse the same configuration for inference later.
network_factory = functools.partial(
    rec_nets.make_recurrent_ppo_networks,
    core_type='gru',
    hidden_size=64,
    policy_hidden_layer_sizes=(64,),
    value_hidden_layer_sizes=(64,),
)

train_kwargs = dict(
    num_timesteps=100_000,
    num_evals=4,
    reward_scaling=10.0,
    episode_length=256,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=16,
    bptt_length=8,
    num_minibatches=8,
    num_updates_per_batch=2,
    discounting=0.97,
    learning_rate=3e-4,
    entropy_cost=1e-2,
    num_envs=512,
    batch_size=256,
    seed=1,
    network_factory=network_factory,
)

print('Starting training...')
start = datetime.now()
make_policy, params, metrics = recurrent_ppo.train(
    environment=env,
    **train_kwargs,
)
elapsed = (datetime.now() - start).total_seconds()
print(f"Training complete in {elapsed:.1f}s")
print({k: float(v) for k, v in metrics.items() if hasattr(v, 'item')})



Save and reload the trained parameters.


In [ ]:
model_dir = '/tmp/recurrent_ppo_pendulum'
os.makedirs(model_dir, exist_ok=True)

from brax.training import checkpoints
checkpoints.save_checkpoint(model_dir, params, step=0, prefix='pendulum_')
params = checkpoints.restore_checkpoint(model_dir, target=None, prefix='pendulum_')



Build an inference function with recurrent state, then visualize a rollout.


In [ ]:
#@title Visualizing a trajectory of the learned inference function

# Rebuild the network so we can create initial recurrent states for inference.
ppo_network = network_factory(env.observation_size, env.action_size)
inference_policy = make_policy(params, deterministic=True)

jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)

rollout = []
rng = jax.random.PRNGKey(seed=42)
core_state = ppo_network.initial_state_fn(batch_size=1)
state = jit_env_reset(rng=rng)

for _ in range(300):
  rollout.append(state.pipeline_state)
  act_rng, rng = jax.random.split(rng)
  actions, _, new_core_state = inference_policy(state.obs, core_state, act_rng)
  state = jit_env_step(state, actions)
  mask = (1.0 - state.done)
  core_state = ppo_network.mask_state_fn(new_core_state, mask)

HTML(html.render(env.sys, rollout))



That is it! You just trained and visualized a recurrent PPO agent on the inverted pendulum.
